# Notebook 01 — mBERT Baseline

## Experiment Overview

This notebook trains a standard mBERT sequence classifier on the original LINCE SA dataset
to establish the **56.6% accuracy baseline**.

The LINCE SA dataset contains Spanish-English code-switched tweets annotated for sentiment
(positive / negative / neutral). We filter for samples with genuine code-switching
(both `lang1` and `lang2` present, ≥ 2 tokens each), giving ~6,300 usable samples.

## Why These Settings?

| Hyperparameter | Value | Rationale |
|---|---|---|
| Model | `bert-base-multilingual-cased` | Pretrained on 104 languages; handles ES/EN mixing |
| Learning rate | `1e-5` | 2e-5 overfits by epoch 3; 1e-5 peaks around epoch 4.5 |
| Epochs | `5` | Validation accuracy plateaus after epoch 4.5 |
| Batch size | `16` | Standard for BERT-base on a single GPU |
| Class weights | `[3.25, 1.96, 1.0]` | Dataset is ~54% positive; penalizes majority-class predictions |

## Result

**Validation accuracy: 56.6%**

Despite 9 follow-up experiments with multi-task learning and architectural changes,
none surpassed this figure — pointing to data quality as the real bottleneck.
See `docs/experiment_log.md` for the full experiment history.

In [ ]:
# Install dependencies (run once)
!pip install transformers datasets accelerate -q

import json
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
from collections import Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from torch.utils.data import Dataset

warnings.filterwarnings("ignore")

# Hyperparameters validated across Experiments 1-14 in experiment_log.md
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128
BATCH_SIZE = 16
LR         = 1e-5   # 1e-5 outperforms 2e-5 on this dataset (Exp. 13 vs. 11)
NUM_EPOCHS = 5      # peak at ~epoch 4.5; validation loss rises beyond this
TEST_SIZE  = 0.2
SEED       = 42
OUTPUT_DIR = "./results/mbert_baseline"

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL  = {0: "negative", 1: "neutral", 2: "positive"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading

We load the LINCE SA dataset directly from Hugging Face and filter for genuine
Spanish-English code-switched samples. A sample qualifies if it has at least
2 English (`lang1`) tokens **and** at least 2 Spanish (`lang2`) tokens.

The raw LINCE SA dataset contains noisy labels — this is what Notebook 02 addresses.
Here we use the **original labels** to establish the unmodified baseline.

In [ ]:
def is_code_switched(lid_labels, min_tokens=2):
    """Return True if the sample contains genuine Spanish-English code-switching."""
    counts = Counter(lid_labels)
    return counts.get("lang1", 0) >= min_tokens and counts.get("lang2", 0) >= min_tokens

def normalize_label(raw):
    """Map LINCE sentiment labels to lowercase strings."""
    int_map = {0: "negative", 1: "positive", 2: "neutral"}
    if isinstance(raw, int):
        return int_map.get(raw)
    s = str(raw).lower()
    return s if s in LABEL2ID else None

print("Loading LINCE SA from Hugging Face ...")
raw_dataset = load_dataset("lince", "sa_spaeng")

records = []
for split in ["train", "validation"]:
    for row in raw_dataset[split]:
        if not is_code_switched(row["lid"]):
            continue
        label = normalize_label(row["sa"])
        if label is None:
            continue
        records.append({"text": " ".join(row["words"]), "label": label})

df = pd.DataFrame(records)
df["label_id"] = df["label"].map(LABEL2ID)

print(f"Usable samples: {len(df):,}")
print("Label distribution:")
for lbl, cnt in df["label"].value_counts().items():
    print(f"  {lbl}: {cnt:,} ({cnt / len(df) * 100:.1f}%)")

# Stratified 80/20 split
train_df, val_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, stratify=df["label"]
)
print(f"\nTrain: {len(train_df):,} | Val: {len(val_df):,}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SentimentDataset(train_df["text"].tolist(), train_df["label_id"].tolist(), tokenizer, MAX_LENGTH)
val_dataset   = SentimentDataset(val_df["text"].tolist(),   val_df["label_id"].tolist(),   tokenizer, MAX_LENGTH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)
print("Data and model ready.")

## 2. Training

We fine-tune mBERT with **class-weighted cross-entropy** to counteract the dataset imbalance
(~54% positive, 19% negative, 27% neutral).

The weights `[3.25, 1.96, 1.0]` (negative, neutral, positive) are inverse-frequency values
validated across Experiments 11–14. They improve minority-class recall without hurting
overall accuracy. The custom `WeightedTrainer` subclass injects these weights into the loss.

In [ ]:
# Class weights reflect inverse label frequency in the training split
# negative : neutral : positive = 3.25 : 1.96 : 1.0
class_weights = torch.tensor([3.25, 1.96, 1.0]).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        loss    = nn.CrossEntropyLoss(weight=class_weights)(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Training ...")
trainer.train()

results      = trainer.evaluate()
final_acc    = results["eval_accuracy"]
train_logs   = trainer.state.log_history
print(f"\nFinal validation accuracy: {final_acc:.4f} ({final_acc * 100:.2f}%)")

## 3. Evaluation

**Result: 56.6% accuracy**

Key observations:
- The model struggles most with **positive / neutral confusion** — mildly positive tweets
  often get classified as neutral and vice versa.
- Validation accuracy peaks around epoch 4.5 and does not improve with more training.
- Nine follow-up multi-task learning experiments (Experiments 1–9) all performed *worse*
  than this number, confirming that the model architecture is not the bottleneck.
- The real issue is data quality — addressed in Notebook 02.

In [ ]:
# Detailed classification report
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=list(ID2LABEL.values())))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(ID2LABEL.values()),
            yticklabels=list(ID2LABEL.values()))
plt.title(f"mBERT Baseline — Accuracy: {final_acc:.3f}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Training and validation loss over steps
train_steps, train_losses = [], []
eval_steps,  eval_losses, eval_accs = [], [], []
for log in train_logs:
    if "loss" in log and "eval_loss" not in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log:
        eval_steps.append(log["step"])
        eval_losses.append(log["eval_loss"])
        eval_accs.append(log["eval_accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_steps, train_losses, label="Train loss")
axes[0].plot(eval_steps,  eval_losses,  label="Val loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(eval_steps, eval_accs, marker="o", color="green")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation accuracy")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Save results summary
os.makedirs(OUTPUT_DIR, exist_ok=True)
summary = {
    "model":    MODEL_NAME,
    "accuracy": float(final_acc),
    "timestamp": datetime.now().isoformat(),
    "hyperparameters": {
        "lr": LR, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
        "class_weights": [3.25, 1.96, 1.0],
    },
}
with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Results saved to {OUTPUT_DIR}/results.json")